# House AI — T4 image-to-3D backend

Uses Stability AI SPAR3D. Run cells in order. The model weights are gated on Hugging Face, so after installation you will need approved access and a read token before inference.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU runtime required"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))


In [ ]:
%cd /content
!rm -rf /content/stable-point-aware-3d
!git clone --depth 1 https://github.com/Stability-AI/stable-point-aware-3d.git /content/stable-point-aware-3d
%cd /content/stable-point-aware-3d
!pip install -q -U setuptools==69.5.1 wheel
import subprocess
subprocess.run(["pip","install","-q","-r","requirements.txt"], check=True)
print("SPAR3D dependencies installed successfully")


In [ ]:
!rm -rf /content/house-ai
!git clone --depth 1 https://github.com/Rohit9605/RohitGundam-house-ai.git /content/house-ai
import os, subprocess, time
os.environ["SPAR3D_DIR"]="/content/stable-point-aware-3d"
server=subprocess.Popen(["python","/content/house-ai/local-world/server.py"],env=os.environ.copy())
time.sleep(3)
print("House AI generation server started")


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, re, time
tunnel=subprocess.Popen(["/content/cloudflared","tunnel","--url","http://127.0.0.1:8787","--no-autoupdate"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None
deadline=time.time()+60
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if line: print(line,end="")
    m=re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com",line)
    if m: url=m.group(0); break
assert url, "Cloudflare tunnel did not start"
print("\\nCOPY THIS URL INTO HOUSE AI:\\n"+url)
print("Keep this Colab runtime connected while generating.")
